**IMPROVEMENTS**

* [] pass the json for validation directly to the model invocation when possible (check if it's supported) i.e. stop to do the validation after the invocation

Self healing strategies:

* [x] increase temperature error after error (already implemented, works sometimes for qwen3)
* [ ] pass the error message when problems in the json are detected to see if it fixes them afterwards (new prompt with schema, invalid JSON, validation error and "Fix the JSON so it passes validation. Do not change values unless required to fix an error.")

In [1]:
import sys
import time
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os
from google import genai

from prompts import SYSTEM_PROMPT
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction, create_base_messages, invoke_model
from api_keys import *

In [2]:
LANGUAGE = "en" #TODO
# MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
# MODEL_ID = "openai/gpt-oss-120b"
MODEL_ID = "gemma-3-27b-it"
LLM_TYPE = "gemma" if "gemma" in MODEL_ID else "groq"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_dynamic-temp_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

FS_DATASET = "NLP-FBK/dyspnea-crf-train"
DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-development"
NUM_FS_EXAMPLES = 3
MAX_RETRIES = 3

In [3]:
os.chdir("..")

In [4]:
# one-time initializations
#TODO move these APIs somewhere else!!!!
if LLM_TYPE=="groq":
    client = Groq(api_key=GROQ_API_KEY)
elif LLM_TYPE=="gemma":
    client = genai.Client(api_key=GOOGLE_API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

base_messages = create_base_messages(CRFModel, SYSTEM_PROMPT, FS_DATASET, LANGUAGE, NUM_FS_EXAMPLES, llm_type="gemma")
base_messages[-1]

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-development, loading from disk...
NLP-FBK/dyspnea-crf-development has 80 documents
Found local cache at data/dyspnea-crf-train, loading from disk...


Content(
  parts=[
    Part(
      text='{"history of recent trauma": "y", "cardiovascular diseases": "y"}'
    ),
  ],
  role='model'
)

In [ ]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

In [ ]:
from time import sleep

#TODO clean this function
def crf_dataset():
    with open(OUTPUT_PATH, "a") as f: #TODO this is append now
        for item in ds:
            doc_id = item["document_id"]
            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            clinical_note = item["clinical_note"]
            success = False
            
            print(f"Processing document: {doc_id}")

            for attempt in range(MAX_RETRIES):
                if "gemma" in MODEL_ID:
                    sleep(20)
                try:
                    # 1. Invoke model (Network/API errors cached here)
                    raw_output = invoke_model(client, MODEL_ID, clinical_note, base_messages.copy(), attempt)
                    
                    # 2. Parse JSON
                    data_dict = json.loads(raw_output)
                    
                    # 3. Validate with Pydantic
                    result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success

                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{raw_output}\n")
                    # Retry
                
                except Exception as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts.")
                
                # Fallback: Create fake instance with all "unknown"
                fake_instance = CRFModel()
                save_prediction(f, doc_id, fake_instance)


crf_dataset()